# Baselines: DummyClassifier e Regressão Logística

Etapa 1 — Passo 4: treinar modelos baseline para estabelecer referência mínima de desempenho antes da MLP.

| Modelo | Propósito |
|---|---|
| `DummyClassifier(strategy='most_frequent')` | Piso trivial: sempre prediz a classe majoritária (não churn) |
| `DummyClassifier(strategy='stratified')` | Piso probabilístico: prediz aleatoriamente respeitando a distribuição do treino |
| `LogisticRegression` | Baseline linear: referência para o ganho da MLP |

**Métricas:** AUC-ROC, PR-AUC, F1 (churn), Accuracy, Brier Score  
**Validação:** StratifiedKFold (5 folds) — obrigatório com dado desbalanceado  
**Tracking:** MLflow com backend SQLite local

## 1. Setup

In [ ]:
import logging
import random
import warnings

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    average_precision_score,
    brier_score_loss,
    classification_report,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_PATH = "../data/raw/telco-churn.csv"
MLFLOW_URI = "sqlite:///../mlruns.db"
EXPERIMENT_NAME = "churn-baselines"
N_FOLDS = 5

plt.rcParams["figure.figsize"] = (10, 4)
sns.set_theme(style="whitegrid")

log.info("Setup concluído. SEED=%d, FOLDS=%d", SEED, N_FOLDS)

## 2. Carregamento e Pré-processamento

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()

# Correções identificadas na EDA
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
df["Churn"] = (df["Churn"] == "Yes").astype(int)

# Remover ID — não é feature preditiva
df = df.drop(columns=["customerID"])

log.info("Dataset carregado: %s", df.shape)
log.info("Churn rate: %.1f%%", df["Churn"].mean() * 100)

print(f"Shape: {df.shape}")
print(f"Churn rate: {df['Churn'].mean():.3f} ({df['Churn'].mean()*100:.1f}%)")
df.head(3)

## 3. Pipeline de Pré-processamento

- **Numéricas** (`tenure`, `MonthlyCharges`, `TotalCharges`): `StandardScaler`
- **Categóricas** (todos os `object` + `SeniorCitizen`): `OneHotEncoder` com `drop='if_binary'` para variáveis binárias

> `TotalCharges` é mantida apesar da multicolinearidade com `tenure` (correlação 0.83). Para a Regressão Logística isso pode inflar coeficientes, mas o objetivo aqui é baseline: resultados reprodutíveis e rastreáveis.

In [ ]:
TARGET = "Churn"
X = df.drop(columns=[TARGET])
y = df[TARGET]

num_features = ["tenure", "MonthlyCharges", "TotalCharges"]
# SeniorCitizen já é int64 (0/1) — tratado como categórico para OHE consistente
cat_features = [c for c in X.columns if c not in num_features]

print(f"Features numéricas ({len(num_features)}): {num_features}")
print(f"Features categóricas ({len(cat_features)}): {cat_features}")
print(f"Total de features: {X.shape[1]}")

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            num_features,
        ),
        (
            "cat",
            OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False),
            cat_features,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

log.info("Preprocessor definido: %d features numéricas, %d categóricas", len(num_features), len(cat_features))

## 4. Função de Avaliação com Validação Cruzada

`StratifiedKFold` garante que cada fold mantenha a proporção de 26.5% de churners, evitando folds com distribuição distorcida que gerariam métricas irreais.

In [ ]:
CV = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)


def evaluate_cv(pipeline: Pipeline, X: pd.DataFrame, y: pd.Series, model_name: str) -> dict:
    """Avalia pipeline com cross-validation estratificado. Retorna dict com métricas."""
    log.info("Avaliando '%s' com %d folds...", model_name, N_FOLDS)

    scoring = {
        "roc_auc": "roc_auc",
        "average_precision": "average_precision",  # PR-AUC
        "f1": "f1",
        "accuracy": "accuracy",
        "brier_score": "neg_brier_score",
    }

    cv_results = cross_validate(
        pipeline,
        X,
        y,
        cv=CV,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1,
    )

    metrics = {
        "roc_auc_mean": cv_results["test_roc_auc"].mean(),
        "roc_auc_std": cv_results["test_roc_auc"].std(),
        "pr_auc_mean": cv_results["test_average_precision"].mean(),
        "pr_auc_std": cv_results["test_average_precision"].std(),
        "f1_mean": cv_results["test_f1"].mean(),
        "f1_std": cv_results["test_f1"].std(),
        "accuracy_mean": cv_results["test_accuracy"].mean(),
        "accuracy_std": cv_results["test_accuracy"].std(),
        "brier_score_mean": -cv_results["test_brier_score"].mean(),
        "brier_score_std": cv_results["test_brier_score"].std(),
    }

    log.info(
        "%s → AUC-ROC=%.3f±%.3f | PR-AUC=%.3f±%.3f | F1=%.3f±%.3f",
        model_name,
        metrics["roc_auc_mean"],
        metrics["roc_auc_std"],
        metrics["pr_auc_mean"],
        metrics["pr_auc_std"],
        metrics["f1_mean"],
        metrics["f1_std"],
    )
    return metrics

## 5. Configuração do MLflow

In [ ]:
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

log.info("MLflow tracking URI: %s", MLFLOW_URI)
log.info("Experimento: %s", EXPERIMENT_NAME)
print(f"MLflow UI: mlflow ui --backend-store-uri {MLFLOW_URI}")

## 6. Baseline 1: DummyClassifier

Estabelece o **piso mínimo de desempenho**. Qualquer modelo real deve superar esses valores consistentemente para ter valor preditivo.

In [ ]:
dummy_majority_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent", random_state=SEED)),
])

dummy_majority_metrics = evaluate_cv(dummy_majority_pipe, X, y, "Dummy (most_frequent)")

params_majority = {
    "model": "DummyClassifier",
    "strategy": "most_frequent",
    "n_folds": N_FOLDS,
    "seed": SEED,
}

with mlflow.start_run(run_name="dummy-most_frequent"):
    mlflow.log_params(params_majority)
    mlflow.log_metrics(dummy_majority_metrics)
    mlflow.sklearn.log_model(dummy_majority_pipe, name="model")
    run_id_dummy_majority = mlflow.active_run().info.run_id

log.info("Run registrado: %s", run_id_dummy_majority)

In [ ]:
dummy_stratified_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="stratified", random_state=SEED)),
])

dummy_stratified_metrics = evaluate_cv(dummy_stratified_pipe, X, y, "Dummy (stratified)")

params_stratified = {
    "model": "DummyClassifier",
    "strategy": "stratified",
    "n_folds": N_FOLDS,
    "seed": SEED,
}

with mlflow.start_run(run_name="dummy-stratified"):
    mlflow.log_params(params_stratified)
    mlflow.log_metrics(dummy_stratified_metrics)
    mlflow.sklearn.log_model(dummy_stratified_pipe, name="model")
    run_id_dummy_stratified = mlflow.active_run().info.run_id

log.info("Run registrado: %s", run_id_dummy_stratified)

## 7. Baseline 2: Regressão Logística

Modelo linear interpretável que estabelece o **teto do linear** — o ganho da MLP deve ser medido contra esse valor.  
`class_weight='balanced'` compensa o desbalanceamento de classes sem precisar de reamostragam.

In [ ]:
lr_pipe = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=SEED,
            solver="lbfgs",
            C=1.0,
        ),
    ),
])

lr_metrics = evaluate_cv(lr_pipe, X, y, "Logistic Regression")

params_lr = {
    "model": "LogisticRegression",
    "C": 1.0,
    "solver": "lbfgs",
    "max_iter": 1000,
    "class_weight": "balanced",
    "n_folds": N_FOLDS,
    "seed": SEED,
    "scaler": "StandardScaler",
    "encoder": "OneHotEncoder(drop=if_binary)",
}

with mlflow.start_run(run_name="logistic-regression-balanced"):
    mlflow.log_params(params_lr)
    mlflow.log_metrics(lr_metrics)
    # Treina no dataset completo para salvar o artefato
    lr_pipe.fit(X, y)
    mlflow.sklearn.log_model(lr_pipe, name="model")
    run_id_lr = mlflow.active_run().info.run_id

log.info("Run registrado: %s", run_id_lr)

## 8. Tabela Comparativa de Resultados

In [ ]:
results = {
    "Dummy (most_frequent)": dummy_majority_metrics,
    "Dummy (stratified)": dummy_stratified_metrics,
    "Logistic Regression": lr_metrics,
}

# Métricas principais para exibição
display_metrics = ["roc_auc_mean", "pr_auc_mean", "f1_mean", "accuracy_mean", "brier_score_mean"]
labels = ["AUC-ROC", "PR-AUC", "F1 (churn)", "Accuracy", "Brier Score"]

rows = []
for model_name, metrics in results.items():
    row = {"Modelo": model_name}
    for metric_key, label in zip(display_metrics, labels):
        mean = metrics[metric_key]
        std = metrics[metric_key.replace("_mean", "_std")]
        row[label] = f"{mean:.3f} ± {std:.3f}"
    rows.append(row)

comparison_df = pd.DataFrame(rows).set_index("Modelo")

print("=" * 80)
print("TABELA COMPARATIVA — BASELINES")
print("=" * 80)
display(comparison_df)
print("\nNota: Brier Score — menor é melhor. Demais métricas — maior é melhor.")

## 9. Visualizações: ROC e PR Curves

Treinamos no dataset completo para gerar as curvas (visualização, não avaliação — o CV acima é a métrica oficial).

In [ ]:
# Treinar todos no dataset completo para visualização
dummy_majority_pipe.fit(X, y)
dummy_stratified_pipe.fit(X, y)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models_for_viz = [
    ("Dummy (most_frequent)", dummy_majority_pipe, "gray"),
    ("Dummy (stratified)", dummy_stratified_pipe, "orange"),
    ("Logistic Regression", lr_pipe, "steelblue"),
]

for name, pipe, color in models_for_viz:
    try:
        y_score = pipe.predict_proba(X)[:, 1]
    except AttributeError:
        # DummyClassifier com most_frequent pode não ter predict_proba útil
        y_score = pipe.predict(X).astype(float)

    roc_auc = roc_auc_score(y, y_score)
    pr_auc = average_precision_score(y, y_score)

    RocCurveDisplay.from_predictions(
        y, y_score,
        name=f"{name} (AUC={roc_auc:.3f})",
        ax=axes[0],
        color=color,
    )
    PrecisionRecallDisplay.from_predictions(
        y, y_score,
        name=f"{name} (AP={pr_auc:.3f})",
        ax=axes[1],
        color=color,
    )

axes[0].set_title("ROC Curve — Baselines")
axes[1].set_title("Precision-Recall Curve — Baselines")

# Linha de referência aleatória no PR curve
churn_rate = y.mean()
axes[1].axhline(y=churn_rate, color="black", linestyle="--", alpha=0.5,
                label=f"Aleatório (P={churn_rate:.2f})")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig("../docs/baselines_curves.png", dpi=120, bbox_inches="tight")
plt.show()
log.info("Curvas salvas em docs/baselines_curves.png")

## 10. Análise de Coeficientes — Regressão Logística

Os coeficientes da Regressão Logística validam se o modelo aprendeu os padrões identificados na EDA.

In [ ]:
feature_names = lr_pipe.named_steps["preprocessor"].get_feature_names_out()
coefs = lr_pipe.named_steps["classifier"].coef_[0]

coef_df = (
    pd.DataFrame({"feature": feature_names, "coef": coefs})
    .assign(abs_coef=lambda d: d["coef"].abs())
    .sort_values("abs_coef", ascending=False)
    .head(20)
    .sort_values("coef")
)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ["tomato" if c > 0 else "steelblue" for c in coef_df["coef"]]
ax.barh(coef_df["feature"], coef_df["coef"], color=colors, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Top 20 Coeficientes — Regressão Logística\n(vermelho = aumenta risco de churn)", fontsize=12)
ax.set_xlabel("Coeficiente (espaço log-odds)")
plt.tight_layout()
plt.savefig("../docs/lr_coefficients.png", dpi=120, bbox_inches="tight")
plt.show()
log.info("Coeficientes salvos em docs/lr_coefficients.png")

## 11. Matriz de Confusão — Regressão Logística

Análise do trade-off entre falso positivo (cliente retido desnecessariamente) e falso negativo (churn não detectado).

In [ ]:
y_pred_lr = lr_pipe.predict(X)
y_proba_lr = lr_pipe.predict_proba(X)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay.from_predictions(
    y, y_pred_lr,
    display_labels=["Não churn", "Churn"],
    ax=axes[0],
    colorbar=False,
    cmap="Blues",
)
axes[0].set_title("Matriz de Confusão\n(Logistic Regression, threshold=0.5)")

ConfusionMatrixDisplay.from_predictions(
    y, y_pred_lr,
    display_labels=["Não churn", "Churn"],
    ax=axes[1],
    colorbar=False,
    cmap="Blues",
    normalize="true",
)
axes[1].set_title("Matriz de Confusão Normalizada\n(Logistic Regression)")

plt.tight_layout()
plt.show()

print("\nClassification Report — Logistic Regression:")
print(classification_report(y, y_pred_lr, target_names=["Não churn", "Churn"]))

## 12. Resumo e Referência para a MLP

Os valores abaixo são os **targets mínimos** que a MLP deve superar para justificar sua complexidade adicional.

In [ ]:
print("=" * 60)
print("REFERÊNCIA PARA A MLP — TARGETS MÍNIMOS")
print("=" * 60)

lr_m = lr_metrics
print(f"\nLogistic Regression (melhor baseline linear):")
print(f"  AUC-ROC  : {lr_m['roc_auc_mean']:.3f} ± {lr_m['roc_auc_std']:.3f}")
print(f"  PR-AUC   : {lr_m['pr_auc_mean']:.3f} ± {lr_m['pr_auc_std']:.3f}")
print(f"  F1       : {lr_m['f1_mean']:.3f} ± {lr_m['f1_std']:.3f}")
print(f"  Accuracy : {lr_m['accuracy_mean']:.3f} ± {lr_m['accuracy_std']:.3f}")
print(f"  Brier    : {lr_m['brier_score_mean']:.3f} ± {lr_m['brier_score_std']:.3f}")

print(f"\nMLP deve superar:")
print(f"  AUC-ROC  > {lr_m['roc_auc_mean']:.3f}")
print(f"  PR-AUC   > {lr_m['pr_auc_mean']:.3f}")
print(f"  F1       > {lr_m['f1_mean']:.3f}")

print("\n" + "=" * 60)
log.info("Baselines concluídos. Métricas registradas no MLflow: %s", EXPERIMENT_NAME)